# Getting Started with MAGEMin in Python

This tutorial demonstrates how to run MAGEMin/Julia modules from Python using [juliacall](https://github.com/JuliaPy/JuliaCall). The workflow is closely related to the examples in the [MAGEMin_C repository](https://github.com/ComputationalThermodynamics/MAGEMin_C.jl).

## 1. Import Required Modules

We use `juliacall` to import MAGEMin from Julia.

In [ ]:
import numpy as np
import juliacall
from juliacall import Main as jl, convert as jlconvert

# Load MAGEMin_C Julia module
MAGEMin_C = juliacall.newmodule("MAGEMin_C")
MAGEMin_C.seval("using MAGEMin_C")

## 2. Example: Predefined Bulk Rock Minimization

We begin by initializing MAGEMin with a predefined bulk rock composition.

In [ ]:
db   = "ig"  # Database options: 'ig', 'mp', etc.
data = MAGEMin_C.Initialize_MAGEMin(db, verbose=False)
test = 0  # Predefined rock (e.g., KLB1)
data = MAGEMin_C.use_predefined_bulk_rock(data, test)
P = 8.0   # Pressure [kbar]
T = 800.0 # Temperature [°C]
out = MAGEMin_C.point_wise_minimization(P, T, data)

## 3. Custom Bulk Composition Minimization

To use a custom bulk rock composition for a single-point minimization, define oxide names and abundances as lists and convert them to Julia vectors.

In [ ]:
data = MAGEMin_C.Initialize_MAGEMin("ig", verbose=False)
P, T = 12.0, 1000.0
# Define oxides and their abundances (wt%)
oxides = ["SiO2", "Al2O3", "CaO", "MgO", "FeO", "Fe2O3", "K2O", "Na2O", "TiO2", "Cr2O3", "H2O"]
Xoxides = jlconvert(jl.Vector[jl.String], oxides)
X = jlconvert(jl.Vector[jl.Float64], [48.43, 15.19, 11.57, 10.13, 6.65, 1.64, 0.59, 1.87, 0.68, 0.0, 3.0])
sys_in = "wt"

In [ ]:
out = MAGEMin_C.single_point_minimization(P, T, data, X=X, Xoxides=Xoxides, sys_in=sys_in)

In [ ]:
out

## 4. Changing Dataset

You can specify a particular dataset when initializing MAGEMin. This is useful for comparing results using different thermodynamic databases.

In [ ]:
data = MAGEMin_C.Initialize_MAGEMin("ig", dataset=636, verbose=False)
out = MAGEMin_C.single_point_minimization(P, T, data, X=X, Xoxides=Xoxides, sys_in=sys_in)
out

## 5. Excluding Phases from Minimization

To remove particular phases from the equilibrium calculation, construct a removal list and pass it as an argument.

In [ ]:
# Example: Remove 'g' phase (e.g., garnet)
rm_solution_phases = ['g']
rm_pure_phases = []
rm_phases = jlconvert(jl.Vector[jl.String], rm_solution_phases + rm_pure_phases)
rm_list = MAGEMin_C.remove_phases(rm_phases, db)
out = MAGEMin_C.single_point_minimization(P, T, data, X=X, Xoxides=Xoxides, sys_in=sys_in, rm_list=rm_list)

## 6. Multi-Point Minimization

For calculations over arrays of pressure and temperature, pass vectors to the minimization function. This enables phase diagram construction or batch processing.

In [ ]:
P = jlconvert(jl.Vector[jl.Float64], [3., 4., 5., 6., 7.])
T = jlconvert(jl.Vector[jl.Float64], [300., 400., 500., 600., 700.])
Xoxides = jlconvert(jl.Vector[jl.String], oxides)
X = jlconvert(jl.Vector[jl.Float64], [48.43, 15.19, 11.57, 10.13, 6.65, 1.64, 0.59, 1.87, 0.68, 0.0, 3.0])
sys_in = "wt"

out = MAGEMin_C.multi_point_minimization(P, T, data, X=X, Xoxides=Xoxides, sys_in=sys_in)

In [ ]:
out

---
- [MAGEMin_C GitHub documentation](https://github.com/ComputationalThermodynamics/MAGEMin_C.jl)
